# Partie 3 — Échantillonnage & Biais  
### Projet EDA — Détection de fraude bancaire

---

## 🎯 Objectifs

- Comprendre les méthodes d'**échantillonnage** et leurs cas d'usage
- Identifier les **biais** spécifiques au dataset de fraude bancaire
- Comprendre l'impact du **déséquilibre de classes** sur les modèles ML

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    sns = None

df = pd.read_csv("creditcard.csv")
print(f"Dataset : {df.shape[0]:,} transactions | {df['Class'].sum():,} fraudes ({df['Class'].mean()*100:.3f}%)")


---
## 3.1 — Méthodes d'échantillonnage

### 🔹 1. Échantillonnage aléatoire simple

Chaque transaction a la **même probabilité** d'être sélectionnée.


In [ ]:
# Échantillonnage aléatoire simple
sample_simple = df.sample(n=10000, random_state=42)

print("Échantillon aléatoire simple (n=10 000)")
print(f"  Fraudes : {sample_simple['Class'].sum()} ({sample_simple['Class'].mean()*100:.3f}%)")
print(f"  Normal  : {(sample_simple['Class']==0).sum()}")
print()
print("⚠️  Le déséquilibre est préservé → très peu de fraudes dans l'échantillon.")


### 🔹 2. Échantillonnage stratifié

On échantillonne proportionnellement par strate (ici la classe `Class`).  
**Garantit la représentation de chaque groupe** dans l'échantillon.


In [ ]:
# Échantillonnage stratifié par Class
sample_stratified = df.groupby("Class", group_keys=False).apply(
    lambda x: x.sample(frac=0.05, random_state=42)
)

print("Échantillon stratifié (5% par classe)")
print(f"  Fraudes : {sample_stratified['Class'].sum()} ({sample_stratified['Class'].mean()*100:.3f}%)")
print(f"  Normal  : {(sample_stratified['Class']==0).sum()}")
print()
print("✅  Proportions préservées — utile pour l'évaluation de modèles.")


### 🔹 3. Oversampling de la classe minoritaire (pour ML)

Face au déséquilibre, on peut **suréchantillonner les fraudes** pour équilibrer les classes.  
Ceci n'est **pas** un biais ici — c'est une technique délibérée pour améliorer la détection.


In [ ]:
# Oversampling manuel (duplication) — illustration
fraud_df  = df[df["Class"] == 1]
normal_df = df[df["Class"] == 0]

# Ratio cible : 10% de fraudes
target_fraud_n = int(len(normal_df) * 0.10)
fraud_oversampled = fraud_df.sample(n=target_fraud_n, replace=True, random_state=42)
df_balanced = pd.concat([normal_df, fraud_oversampled]).sample(frac=1, random_state=42)

print("Dataset avec oversampling (10% fraudes)")
print(f"  Total       : {len(df_balanced):,}")
print(f"  Fraudes     : {df_balanced['Class'].sum():,} ({df_balanced['Class'].mean()*100:.1f}%)")
print()
print("⚠️  L'oversampling par duplication peut créer du surapprentissage.")
print("   Préférer SMOTE (Synthetic Minority Over-sampling Technique) en pratique.")


In [ ]:
# Visualisation comparative
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
configs = [
    (df, "Dataset original", "#EF4444"),
    (sample_stratified, "Stratifié (5%)", "#3B82F6"),
    (df_balanced, "Avec oversampling", "#10B981"),
]
for ax, (data, title, color) in zip(axes, configs):
    counts = data["Class"].value_counts().rename({0: "Normal", 1: "Fraude"})
    counts.plot(kind="bar", ax=ax, color=["#3B82F6", "#EF4444"], edgecolor="white")
    ax.set_title(title)
    ax.set_xticklabels(["Normal", "Fraude"], rotation=0)
    ax.set_ylabel("Nombre de transactions")
plt.suptitle("Comparaison des stratégies d'échantillonnage", fontsize=13)
plt.tight_layout()
plt.show()


---
## 3.2 — Types de biais dans le dataset fraude

Les biais peuvent **fausser vos analyses et vos modèles ML**. Voici les principaux à connaître.

### ⚠️ 1. Biais de sélection

Le dataset couvre **une période précise** (2 jours de transactions). Les patterns de fraude peuvent évoluer :
- nouvelles techniques de fraude non présentes
- comportements saisonniers non capturés


### ⚠️ 2. Biais de déséquilibre de classes (Class Imbalance)

Avec seulement **0.17% de fraudes**, un modèle naïf qui prédit toujours "Normal" atteint 99.83% d'accuracy mais détecte **zéro fraude**.


In [ ]:
# Illustration du piège de l'accuracy
n_total = len(df)
n_fraud = df["Class"].sum()
naive_accuracy = (n_total - n_fraud) / n_total

print(f"Modèle naïf (prédit toujours Normal=0) :")
print(f"  Accuracy  : {naive_accuracy:.4f}  ({naive_accuracy*100:.2f}%)")
print(f"  Fraudes détectées : 0 / {n_fraud}")
print()
print("→ Métriques adaptées au déséquilibre :")
print("  • Précision, Rappel, F1-score")
print("  • AUC-ROC, AUC-PR (Precision-Recall)")
print("  • Matrice de confusion")


### ⚠️ 3. Biais d'anonymisation (PCA)

Les variables `V1`–`V28` ont été transformées par **PCA pour anonymisation**.  
Cela rend l'interprétation métier impossible — on ne sait pas quelle composante correspond à quoi.


### ⚠️ 4. Biais temporel (Data Leakage)

Si on utilise des données futures pour prédire des fraudes passées → **fuite de données**.  
Il faut toujours respecter l'ordre chronologique (via `Time`) lors du découpage train/test.


In [ ]:
# Bonne pratique : split chronologique
df_sorted = df.sort_values("Time")
split_idx = int(len(df_sorted) * 0.8)

train = df_sorted.iloc[:split_idx]
test  = df_sorted.iloc[split_idx:]

print("Split chronologique (80% train / 20% test) :")
print(f"  Train : {len(train):,} transactions  | fraudes : {train['Class'].sum():,} ({train['Class'].mean()*100:.3f}%)")
print(f"  Test  : {len(test):,}  transactions  | fraudes : {test['Class'].sum():,} ({test['Class'].mean()*100:.3f}%)")
print()
print("✅  Respect de l'ordre temporel → pas de fuite de données.")


---
## 3.3 — Analyse des valeurs manquantes et doublons


In [ ]:
# Vérification des valeurs manquantes
missing = df.isnull().sum()
print("Valeurs manquantes par colonne :")
if missing.sum() == 0:
    print("  ✅  Aucune valeur manquante dans ce dataset.")
else:
    print(missing[missing > 0])

# Doublons
n_duplicates = df.duplicated().sum()
print(f"\nDoublons : {n_duplicates}")
if n_duplicates > 0:
    print(f"  ⚠️  {n_duplicates} lignes dupliquées à supprimer.")
    df = df.drop_duplicates()
    print(f"  → Dataset après suppression : {len(df):,} lignes")


---
## 📋 Récapitulatif — Partie 3

| Biais | Impact | Solution |
|-------|--------|----------|
| Déséquilibre de classes | Accuracy trompeuse | SMOTE, class_weight, AUC-PR |
| Sélection temporelle | Généralisation limitée | Données récentes, validation temporelle |
| PCA (anonymisation) | Non-interprétabilité | Accepter les composantes comme features |
| Data leakage | Suroptimisme du modèle | Split chronologique obligatoire |
| Doublons | Surapprentissage | Déduplication avant modélisation |
